# Lab 1.3 &mdash; The create_agent On-Ramp

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Judge a tool description the way the model does -- and write one that passes
- Assemble a create_agent configuration and run it against the case file
- Compare the framework agent with your hand-rolled loop on the same briefs

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Builds on Labs 1.1 and 1.2.** Same tools, same case file. What changes is who owns
> the loop: you, or `create_agent`.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
LLM_MODEL    = os.environ.get("LAB_LLM_MODEL")    or os.environ.get("OPENAI_MODEL")
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.2
# These are the tools you wrote in Lab 1.2. Nothing to fill in -- they are here so this
# notebook runs on its own. Note the docstrings: they name the case AND the boundary.

def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}
print("carried forward:", ", ".join(TOOLS))

## Concept

`create_agent(model=..., tools=..., prompt=...)` **is** the loop you wrote in Lab 1.1, prebuilt.
Nothing is hidden: the model decides, a tool runs, the result comes back, repeat until it stops.

> **Naming.** In LangChain 1.x it is `create_agent`. The older `create_react_agent` name is
> everywhere online and is **not** what this course uses.

## Section 1 &mdash; A tool description is an instruction

The model picks a tool using its description and nothing else. Rather than take that on faith,
write the checker: what must a description contain to be choosable?

In [ ]:
def describes_well(doc: str) -> bool:
    """True when a tool docstring gives the model enough to choose correctly.

    A usable description does three things:
      1. says what the tool returns,
      2. names the situation it is FOR ("use when ..."),
      3. names at least one situation it is NOT for -- the boundary.
    """
    if not doc:
        return False
    text = doc.lower()
    says_return = "return" in text
    says_when = BLANK                  # TODO: does it name the situation it is for?
    says_boundary = BLANK              # TODO: does it name a situation it is NOT for?
    return says_return and says_when and says_boundary

In [ ]:
# --- Self-check: Section 1
_weak = "Gets data."
_strong = ("Return the ledger record for one payment reference. Use when you need the status of a "
           "specific payment. Not for searching across payments.")

check("a vague description is rejected", lambda: describes_well(_weak) is False)
check("a specific description is accepted", lambda: describes_well(_strong) is True,
      "look for a 'use when' phrase and a 'not for' boundary")
check("an empty description is rejected", lambda: describes_well("") is False)
check("a description with no boundary is rejected",
      lambda: describes_well("Return the record. Use when you need a payment.") is False,
      "the boundary is what stops the model reaching for the wrong tool")

## Section 2 &mdash; Rewrite the weak tool

`get_data` below is the kind of tool that quietly ruins an agent. Rewrite its docstring so it
passes your own checker &mdash; and note that you are not changing a single line of logic.

In [ ]:
def get_data(ref: str) -> str:
    """BLANK"""                        # TODO: rewrite so describes_well() passes. Logic stays as-is.
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})

In [ ]:
# --- Self-check: Section 2
check("the rewritten docstring passes your checker",
      lambda: describes_well(get_data.__doc__) is True,
      "it needs a return clause, a 'use when', and a boundary")
check("the docstring names the concrete input format",
      lambda: "PMT-" in (get_data.__doc__ or ""),
      "an example reference removes a whole class of malformed calls")
check("the behaviour is unchanged", lambda: "INSUFFICIENT_FUNDS" in get_data("PMT-1002"))

## Section 3 &mdash; The agent configuration

`create_agent` takes four things. Three of them map straight onto the blocks from Lab 1.2.
Assemble the configuration as plain data first, so the shape is checkable before anything runs.

In [ ]:
def build_config() -> dict:
    """Assemble the create_agent configuration as plain data."""
    system_prompt = BLANK              # TODO: a standing instruction. It must (a) give the agent its
                                     # role, and (b) tell it never to act on a payment that policy
                                     # reserves for a human. Mention "human" explicitly.
    return {
        "model": LLM_MODEL,          # block 1: the brain
        "tools": BLANK,                # TODO: a list of the two tool FUNCTIONS carried forward above
        "prompt": system_prompt,     # the standing instruction
        "max_steps": 6,              # the budget from Lab 1.1
    }

In [ ]:
# --- Self-check: Section 3   (structure only -- no model call)
check("the config carries exactly the two tools",
      lambda: len(build_config()["tools"]) == 2)
check("both tools are callables with docstrings",
      lambda: all(callable(t) and (t.__doc__ or "").strip() for t in build_config()["tools"]))
check("the prompt gives the agent a role",
      lambda: len(build_config()["prompt"]) > 40)
check("the prompt defers irreversible decisions to a human",
      lambda: "human" in build_config()["prompt"].lower(),
      "an approval boundary belongs in the standing instruction, not in each request")
check("the budget survived from Lab 1.1", lambda: build_config()["max_steps"] == 6)

## Run it for real

Now hand your configuration to `create_agent`. The `@tool` decorator turns a plain function into
something the model can call &mdash; it reads the signature and the docstring you just wrote.

In [ ]:
if llm_ready():
    try:
        from langchain.tools import tool
        from langchain.agents import create_agent

        cfg = build_config()
        tools = [tool(f) for f in cfg["tools"]]
        agent = create_agent(model=get_llm(), tools=tools, prompt=cfg["prompt"])

        result = agent.invoke({"messages": [("human", "Why is PMT-1005 held, and what do we do?")]})
        for m in result["messages"]:
            kind = getattr(m, "type", "?")
            body = str(getattr(m, "content", ""))[:300]
            calls = getattr(m, "tool_calls", None)
            print(f"[{kind}] {body}" + (f"  -> calls: {[c['name'] for c in calls]}" if calls else ""))
    except ImportError as exc:
        print(f"LangChain not importable here ({exc}). The graded cells above do not need it.")
    except NameError:
        print("(fill in the blanks above, then re-run this cell)")
    except Exception as exc:
        print(f"<agent run failed: {type(exc).__name__}: {exc}>")

### Read the trace

Count the messages. Every `[ai]` with `tool_calls` is one turn of the loop you wrote by hand in
Lab 1.1; every `[tool]` is the observation coming back. `create_agent` saved you the plumbing and
nothing else &mdash; which is exactly why it stops being enough the moment you need to *see* or
*steer* that state. That is Module 3.

Check the last message: with `SANCTIONS_REVIEW` the agent should defer to a human. If it proposed
an action instead, your standing instruction was not firm enough &mdash; and no amount of model
capability fixes an instruction that never said it.

In [ ]:
score()

## Your turn

1. Degrade `get_data`'s docstring back to `"Gets data."`, re-run the live cell, and count the
   tool calls. That difference is the entire content of the Day 2 tool-description lab.
2. Add a third tool that *overlaps* with `lookup_payment` (say `get_payment_status`). Which
   description does the model prefer, and what does that tell you about writing boundaries?